In [0]:
# RESTART PYTHON KERNEL
# This clears the module cache so the fixed path_resolver.py loads with
# Unity Catalog Volume paths instead of the old DBFS paths.
#
# IMPORTANT: After running this cell, run Cell 2 (not Cell 1).

dbutils.library.restartPython()

In [0]:
# 1. Install required library for Serverless
%pip install delta-spark==3.0.0

# 2. Add the repo to the system path (Replace [YOUR_EMAIL] with the email seen in your screenshot)
import sys
import os
# Based on your screenshot, your email is shareitalike@gmail.com
repo_path = "/Workspace/Repos/shareitalike@gmail.com/Lakehouse-Data-Quality-Framework"
sys.path.append(repo_path)

# 3. Import and Run
from pipelines.orchestrator import run_full_pipeline
from config.pipeline_configs import PipelineConfig

config = PipelineConfig.default()
results = run_full_pipeline(config, inject_issues=True)

# 4. Display Results
print(f"✅ Success! Silver Quarantined: {results['silver']['quarantine_count']}")


In [0]:
# --- DATABRICKS SERVERLESS ENTRY POINT (FIXED) ---
import sys
import os
import importlib

# 1. Setup the Path (CORRECTED: use /Workspace/Users/ not /Workspace/Repos/)
repo_path = "/Workspace/Users/shareitalike@gmail.com/Lakehouse-Data-Quality-Framework"
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

print("✅ FIXED: Updated utils/path_resolver.py to use Unity Catalog Volumes")
print("  Changed: /user/.../lakehouse_dq → /Volumes/dev/default/lakehouse_dq")
print()

# 2. Force reload modules to pick up the file changes
if 'utils.path_resolver' in sys.modules:
    importlib.reload(sys.modules['utils.path_resolver'])
if 'config.pipeline_configs' in sys.modules:
    importlib.reload(sys.modules['config.pipeline_configs'])
if 'pipelines.orchestrator' in sys.modules:
    importlib.reload(sys.modules['pipelines.orchestrator'])

print("✅ Reloaded modules to pick up source file changes\n")

# 3. Import (after reload)
from pipelines.orchestrator import run_full_pipeline
from config.pipeline_configs import PipelineConfig

# 4. Verify paths are correct
config = PipelineConfig.default()
print(f"Base path: {config.paths.base_path}")
print(f"Bronze:    {config.paths.bronze_raw}")
print(f"Metrics:   {config.paths.observability_metrics}\n")

if "/Volumes/" not in config.paths.base_path:
    print("⚠️  WARNING: Still using DBFS paths! Module reload may not have worked.")
    print("   Try: Detach and reattach compute, or restart Python kernel.\n")

# 5. RUN PIPELINE
results = run_full_pipeline(config, inject_issues=True)

# 6. SUMMARY
print("\n" + "="*40)
print("🎯 EXECUTION SUMMARY")
print("="*40)

for layer in ["bronze", "silver", "gold"]:
    info = results.get(layer, {})
    status = info.get("status", "NOT RUN")
    
    if status == "SUCCESS":
        count = info.get("orders_count") or info.get("record_count") or info.get("daily_revenue_rows")
        q_count = info.get("quarantine_count", 0)
        print(f"✅ {layer.upper()}: Success ({count} records, {q_count} quarantined)")
    else:
        error = info.get("error", "Unknown error")
        print(f"❌ {layer.upper()}: Failed - {error}")

print("="*40)
print(f"Master Run ID: {results['master_run_id']}")